# 第 6 章：DQN + 函数逼近 —— 进入深度 RL

> **Ch01-05 的所有环境都是离散状态**（GridWorld 的整数索引、Random Walk 的位置、bandit 的固定均值）。
> 现实问题几乎都是**连续状态**——围棋棋盘配置 $10^{170}$、机器人关节角度 $\in \mathbb{R}^n$、Atari 像素 $10^{6+}$ 维。
> 表格方法在这种规模下**完全失效**——表项数爆炸，根本存不下。
>
> **本章核心**：用神经网络 $Q(s, a; \theta)$ 近似 Q-table，把 Ch05 的 Q-learning 升级到 **DQN**（Deep Q-Network）。

## 学习目标

1. 理解 **函数逼近**（function approximation）的动机与三种架构
2. 推导**梯度 TD** 损失函数和**半梯度**概念
3. 掌握**致命三件套**（Deadly Triad）——为什么 DQN 训练不稳定
4. 实现完整 **DQN**（experience replay + target network + ε 衰减）
5. 理解 **Double DQN** 和 **Dueling DQN** 两个核心改进
6. 在自研 **CartPoleLite** 环境上从零训练一个能撑住 500 步的 agent

## 承接 Phase 1

Ch05 的 Q-learning 已经是完整的无模型控制算法（§5.3 off-policy、§5.7-5.8 maximization bias 与 Double Q），但它建立在**表格**上。本章把 Q 搬进神经网络，一路上会直面四个新问题：

- **经验回放（Experience Replay）**：神经网络不喜欢相关样本——怎么打破时序相关性？
- **目标网络（Target Network）**：TD target 依赖网络自己——怎么避免"追逐移动目标"？
- **Double DQN**：Ch05 §5.8 的解耦思想在 Deep RL 里的版本
- **致命三件套（Deadly Triad）**：函数逼近 + 自举 + off-policy 同时出现的发散风险

本章逐一给出解法。

In [ ]:
# 常规设置：找项目根、载入库
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'rlenvs').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from rlenvs import CartPoleLite
from utils import set_seed
from utils.networks import QNetwork, DuelingQNetwork, make_mlp
from utils.replay import ReplayBuffer
from utils.dqn_utils import (
    linear_epsilon_schedule,
    hard_update, polyak_update,
    epsilon_greedy_action, dqn_update_step,
)
from utils.torch_utils import get_device, count_parameters, grad_stats

set_seed(42)
torch.manual_seed(42)
np.random.seed(42)

DEVICE = get_device()
print(f"PyTorch: {torch.__version__}, device = {DEVICE}")

## 6.1 表格方法的极限：为什么需要函数逼近

### 6.1.1 表格方法的"状态空间爆炸"

Ch01-05 用 $Q$ 表（二维数组 $|\mathcal{S}| \times |\mathcal{A}|$）存所有 $(s, a)$ 的价值。这要求：
- 状态可枚举（有限离散）
- 每个 $(s, a)$ 都被多次访问（才能估准）

现实中：
| 任务 | 状态空间大小 | 表格可行？ |
|---|---|---|
| GridWorld 5x5 | 25 | ✓ |
| 国际象棋 | $\sim 10^{47}$ | ✗ |
| 围棋 | $\sim 10^{170}$ | ✗ |
| Atari（像素） | $256^{210 \times 160} \approx 10^{10^5}$ | ✗ |
| 机器人关节 | $\mathbb{R}^{20}$（连续） | ✗ |
| LLM token 序列 | $|\text{vocab}|^{\text{ctx}}$ | ✗ |

**核心问题**：表格方法**无法泛化**——它对每个 $(s, a)$ 独立学习，即使两个状态"很像"（如 GridWorld 的相邻格），价值估计也互不影响。

### 6.1.2 函数逼近的洞察

**解决方案**：用参数化函数 $Q(s, a; \theta)$ 近似 $Q$ 表。$\theta \in \mathbb{R}^d$ 是参数（$d$ 远小于表格大小）。

$$
\text{表格：} Q \in \mathbb{R}^{|\mathcal{S}| \times |\mathcal{A}|} \quad \longrightarrow \quad \text{函数：} Q(\cdot, \cdot; \theta),\ \theta \in \mathbb{R}^d
$$

**好处**：
1. **存储 $O(d)$** 而非 $O(|\mathcal{S}||\mathcal{A}|)$
2. **泛化**：相近的 $(s, a)$ 通过共享参数得到相近的 $Q$ 值
3. **连续状态**：$\mathbb{R}^n \to \mathbb{R}$ 天然支持

**代价**：
1. 失去精确性——$Q$ 表能精确存每个值，神经网络只能逼近
2. 引入新麻烦——**致命三件套**（6.4 节）

### 6.1.3 从 Ch05 的 Q-learning 谈起

Ch05 Q-learning 的更新：
$$
Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \big[ R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a') - Q(S_t, A_t) \big]
$$

把 $Q$ 表换成神经网络 $Q(\cdot, \cdot; \theta)$：把"朝 target 走一小步"换成"沿梯度方向调参"。下一节我们推导具体怎么做。

## 6.2 三种逼近架构 & 为什么 Q 直接输出最常用

给定一个神经网络 $f_\theta$，有三种方式把它接入 RL：

### 6.2.1 架构 A：状态 → Q 值（最常用，DQN 用这个）

$$
f_\theta: \mathcal{S} \to \mathbb{R}^{|\mathcal{A}|}, \qquad Q(s, a; \theta) = f_\theta(s)_a
$$

网络一次输出所有动作的 Q 值，`argmax` 直接拿到。

**优点**：
- 单次前向给所有 $a$ 的 Q，`max`/`argmax` $O(1)$（其他架构需要 $|\mathcal{A}|$ 次前向）
- 与 Q-learning 的 `max_a'` 完美匹配

**缺点**：要求**离散动作**（连续动作空间无法枚举）。这就是为什么策略梯度（Ch07）需要另一种架构。

### 6.2.2 架构 B：(state, action) → scalar Q

$$
f_\theta: \mathcal{S} \times \mathcal{A} \to \mathbb{R}, \qquad Q(s, a; \theta) = f_\theta(s, a)
$$

**优点**：支持连续动作（输入连续 $a$，输出标量 Q）——DQN 的"反传统"变种。
**缺点**：每次 `argmax_a` 需要 $|\mathcal{A}|$ 次前向，慢。

### 6.2.3 架构 C：状态 → 策略分布（Ch07 用）

$$
\pi_\theta: \mathcal{S} \to \Delta^{|\mathcal{A}|}, \qquad \pi_\theta(a|s) = f_\theta(s)_a
$$

直接参数化**策略**而不是 Q 值。这是**策略梯度**（Ch07）的起点，不在本章范围。

### 6.2.4 为什么 DQN 选架构 A

离散动作 + 一次前向给所有 Q → 极致工程效率。Mnih et al. 2015 的 DQN 原文用架构 A 在 Atari 上击败人类；后续 Rainbow、IQN、QR-DQN 全部沿用。**本章和后续 Ch08 Actor-Critic 中 actor 网络都用架构 A 思想（连续动作时改用策略参数化，见 Ch07）**。

## 6.3 梯度 TD 与半梯度（**核心推导**）

### 6.3.1 目标函数

我们希望 $Q(s, a; \theta) \approx Q^\pi(s, a)$（贝尔曼方程的真值）。**均方 Bellman 误差**：

$$
J(\theta) = \mathbb{E}\big[\big(\underbrace{R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a'; \theta^-)}_{\text{Bellman target}} - \underbrace{Q(S_t, A_t; \theta)}_{\text{prediction}}\big)^2\big]
$$

这里引入了一个新记号 $\theta^-$——**target network** 的参数（与 $\theta$ 区分；先用着，6.6 节解释为什么需要）。

### 6.3.2 全梯度 vs 半梯度

求 $\theta$ 梯度时，问题来了：**target 里也有 $\theta$ 吗？**

- **全梯度**：把 target 也当成 $\theta$ 的函数求导——梯度上包含 $-\gamma \nabla Q(S_{t+1}, a'; \theta)$ 项
- **半梯度**（semi-gradient）：把 target 当**常数**（$\theta^-$ "冻结"），只对 prediction 求导

**为什么用半梯度？**

1. **计算简单**：少算一次反向传播
2. **更稳定**：全梯度会引入"自己追自己"的问题（prediction 的变化同时改变 target）
3. **数学上自洽**：如果 $\theta^- $ 是 $\theta$ 的滞后副本（与 $\theta$ 弱相关），把 target 当常数是合理近似

**实践中 DQN 一律用半梯度 + target network**（6.6 节）。

### 6.3.3 半梯度下降的更新规则

对单个 transition $(s, a, r, s', \text{done})$，损失

$$
L(\theta) = \big(r + \gamma \max_{a'} Q(s', a'; \theta^-) \cdot (1 - \text{done}) - Q(s, a; \theta)\big)^2
$$

（$(1-\text{done})$ 让终止态 target 退化为 $r$，复用 Ch04/05 的约定）

定义 TD target：$\hat{G} = r + \gamma \max_{a'} Q(s', a'; \theta^-) \cdot (1 - \text{done})$

半梯度（target 不参与求导）：

$$
\nabla_\theta L = -2 \big(\hat{G} - Q(s, a; \theta)\big) \nabla_\theta Q(s, a; \theta)
$$

梯度下降：

$$
\theta \leftarrow \theta + \alpha \big(\hat{G} - Q(s, a; \theta)\big) \nabla_\theta Q(s, a; \theta)
$$

形式上和 Ch04 的 TD(0) 更新规则**一模一样**，只是 $\nabla Q$ 代替了指示函数 $\mathbb{1}[S_t = s]$——这就是"梯度 TD"。

### 6.3.4 PyTorch 实现（半梯度自动）

在 PyTorch 里，**target network 输出放在 `torch.no_grad()` 块里**就实现了半梯度：

```python
# 半梯度（推荐）——target 不参与求导
with torch.no_grad():
    target = r + gamma * target_net(s_next).max(dim=1)[0]
loss = ((target - online_net(s).gather(1, a)) ** 2).mean()
loss.backward()  # 只对 online_net 的参数求梯度
```

In [ ]:
# 演示：半梯度 vs 全梯度的代码区别
import torch
import torch.nn as nn

torch.manual_seed(0)

# 极简 Q 网络：输入 4 维状态，输出 2 维 Q 值
Q = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 2))
Q_target = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 2))
Q_target.load_state_dict(Q.state_dict())  # 初始相同

s = torch.randn(1, 4)
a_idx = 1  # 选第 1 个动作
r = torch.tensor([1.0])
s_next = torch.randn(1, 4)
gamma = 0.99

# 半梯度（推荐）
with torch.no_grad():
    target = r + gamma * Q_target(s_next).max(dim=1)[0]
q_sa = Q(s)[0, a_idx]  # 直接索引 [0, a]
loss_semi = ((target - q_sa) ** 2).mean()
loss_semi.backward()
print(f"半梯度 loss = {loss_semi.item():.4f}")
print(f"  Q 参数梯度 norm = {sum(p.grad.norm()**2 for p in Q.parameters())**0.5:.4f}")

# 重置梯度，重新构造计算图
Q.zero_grad()
q_sa = Q(s)[0, a_idx]

# 全梯度（不推荐——target 也参与求导）
target_full = r + gamma * Q(s_next).max(dim=1)[0]  # 没有 no_grad！
# 注意：max 返回 (values, indices)，对 values 求导仍可微
loss_full = ((target_full - q_sa) ** 2).mean()
loss_full.backward()
print(f"全梯度 loss = {loss_full.item():.4f}（与半梯度相同，因为初始 θ = θ⁻）")
print(f"  Q 参数梯度 norm = {sum(p.grad.norm()**2 for p in Q.parameters())**0.5:.4f}")
print(f"  → 全梯度多了一项 -∂target/∂θ，方向可能抵消也可能叠加，量级不一定更大")

# ---------------------------------------------------------------------------
# 数值验证：半梯度的 autograd vs 有限差分
# 教材传统——凡是"推导出的梯度"都要用数值方法对一遍。
# 这里验证 §6.3.3 的更新式：∇L = -2(target - Q)·∇Q（target 为常数）
# ---------------------------------------------------------------------------
torch.manual_seed(1)
Q2 = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2)).double()  # float64：差分精度需要
s2, a2 = torch.randn(1, 4, dtype=torch.float64), 1
r2 = torch.tensor([0.5], dtype=torch.float64)
sn2, gamma = torch.randn(1, 4, dtype=torch.float64), 0.99

# 1) TD target 的贝尔曼备份手算核对（numpy vs torch）
with torch.no_grad():
    tgt2 = r2 + gamma * Q2(sn2).max(dim=1)[0]
q_next_np = Q2(sn2).detach().numpy()[0]
manual_target = r2.item() + gamma * float(q_next_np.max())
print(f"\n[验证 1] TD target: torch={tgt2.item():.6f}  手算={manual_target:.6f}  "
      f"{'✓' if abs(tgt2.item() - manual_target) < 1e-9 else '✗'}")

# 2) 半梯度 autograd vs 有限差分（对每个参数 ±ε 扰动）
def loss_of_theta():
    return ((tgt2 - Q2(s2)[0, a2]) ** 2).mean()   # tgt2 已 detach → 半梯度

loss_of_theta().backward()
grad_analytic = torch.cat([p.grad.flatten() for p in Q2.parameters()])

eps = 1e-6
grad_fd = torch.zeros_like(grad_analytic)
i = 0
with torch.no_grad():   # 差分过程不需要梯度；p.data 允许原位扰动参数
    for p in Q2.parameters():
        flat = p.data.view(-1)
        for k in range(flat.numel()):
            old = flat[k].item()
            flat[k] = old + eps; lp = loss_of_theta().item()
            flat[k] = old - eps; lm = loss_of_theta().item()
            flat[k] = old
            grad_fd[i] = (lp - lm) / (2 * eps)
            i += 1
err = (grad_analytic - grad_fd).abs().max()
print(f"[验证 2] 半梯度 autograd vs 有限差分: max|Δ|={err:.2e} "
      f"({'✓ 一致' if err < 1e-8 else '✗'})——§6.3.3 的更新式成立")

## 6.4 致命三件套（Deadly Triad）

### 6.4.1 定义

Deep RL 训练不稳定的根源是三个东西同时出现：

1. **Bootstrapping**：用估计更新估计（Ch04 TD：$\hat G = R + \gamma V(S')$）
2. **Off-policy**：用 behavior 策略采的数据训练 target 策略（Ch05 Q-learning）
3. **Function approximation**：神经网络参数化（本章 6.1）

**Sutton & Barto 第 11 章把这叫"Deadly Triad"**——三者任意两个都还行，三个同时出现训练就**可能发散**。

### 6.4.2 为什么三者合一会发散

**直觉**：
- Bootstrapping 让 target 依赖当前 $\theta$
- Function approximation 让 $\theta$ 的更新通过共享参数影响**所有状态**的 Q（不只是当前 $s$）
- Off-policy 让采的 distribution 和学的 distribution 不一致

合起来：$\theta$ 变化 → target 变化 → 通过共享参数**其它状态的 target 也变化** → $\theta$ 进一步变化 → 正反馈循环。

**Tsitsiklis & Van Roy 1997** 严格证明了半梯度 off-policy TD 在函数逼近下**不保证收敛**——$\theta$ 可能在有限步内发散到无穷。

### 6.4.3 数值演示：训练不稳定的 DQN

让我们做个反面实验：去掉 target network、去掉 replay buffer，看 DQN 是否真的会发散。

In [ ]:
# 反面实验：naive DQN（无 target net、无 replay），看是否真的发散
import copy

torch.manual_seed(0)
env = CartPoleLite(seed=0)

# 单个 Q 网络，没有 target network，没有 replay buffer
Q_naive = QNetwork(state_dim=4, n_actions=2, hidden_dims=[64, 64])
opt = torch.optim.Adam(Q_naive.parameters(), lr=1e-3)
gamma = 0.99

losses = []
q_means = []

for episode in range(50):
    s = env.reset()
    done = False
    ep_loss = []
    while not done:
        # ε-greedy
        epsilon = 0.1
        if np.random.random() < epsilon:
            a = np.random.randint(2)
        else:
            with torch.no_grad():
                a = int(Q_naive(torch.as_tensor(s, dtype=torch.float32).unsqueeze(0)).argmax())

        s_next, r, done, _ = env.step(a)

        # 关键：target 也来自同一个 Q_naive（全梯度 + 无 target net）
        s_t = torch.as_tensor(s, dtype=torch.float32).unsqueeze(0)
        sn_t = torch.as_tensor(s_next, dtype=torch.float32).unsqueeze(0)

        # 没有 no_grad——target 参与求导（全梯度）
        with torch.no_grad():
            target = r + gamma * Q_naive(sn_t).max(dim=1)[0] * (0 if done else 1)
        q_sa = Q_naive(s_t)[0, a]
        loss = (target - q_sa) ** 2

        opt.zero_grad()
        loss.backward()
        opt.step()

        ep_loss.append(loss.item())
        s = s_next

    losses.append(np.mean(ep_loss))
    with torch.no_grad():
        # 采样一组固定状态看 Q 值大小
        test_s = torch.randn(32, 4)
        q_means.append(Q_naive(test_s).abs().mean().item())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(losses)
axes[0].set_xlabel('episode')
axes[0].set_ylabel('mean loss per episode')
axes[0].set_title('Naive DQN loss（可能震荡或爆炸）')
axes[0].grid(alpha=0.3)

axes[1].plot(q_means)
axes[1].set_xlabel('episode')
axes[1].set_ylabel('|Q| 平均值')
axes[1].set_title('Q 值演化（健康的训练应保持在合理量级）')
axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print(f"注意：50 episodes 后 |Q| 平均 = {q_means[-1]:.2f}")
print(f"理论上 Q 应该接近 E[Σ γ^t r_t]，对 CartPole max 500 步 reward 1.0，Q ≈ (1-0.99^500)/(1-0.99) ≈ 99")

## 6.5 Experience Replay：打破时序相关性

### 6.5.1 为什么需要

DQN 用 SGD 更新，**SGD 假设样本 i.i.d.**。但 agent 采的 transition $(S_t, A_t, R_{t+1}, S_{t+1})$ 和 $(S_{t+1}, A_{t+1}, R_{t+2}, S_{t+2})$ **高度相关**（共享状态）——这是时间相关性。

直接用这种序列数据训练神经网络，会导致：
1. **过拟合到局部轨迹**（学到的是"特定 episode 的特定顺序"而不是 $Q^*$）
2. **梯度方向噪声大**（连续样本的梯度方向几乎平行）

### 6.5.2 解决方案

把 transition 存到 **replay buffer** 里，每次训练**随机均匀采样** mini-batch。这样：

- **打破时间相关性**：batch 里是不同 episode、不同时间点的混合样本
- **样本效率**：每条 transition 可以被多次学习（on-policy 只能用一次）
- **稳定性**：梯度噪声降低

### 6.5.3 实现

我们的 `ReplayBuffer` 在 `utils/replay.py` 里。核心是循环覆盖 + 均匀采样。

In [ ]:
# ReplayBuffer 演示
buf = ReplayBuffer(capacity=1000, state_dim=4)
env = CartPoleLite(seed=0)

# 用随机策略填 buffer
for episode in range(20):
    s = env.reset()
    done = False
    while not done:
        a = np.random.randint(2)
        s_next, r, done, _ = env.step(a)
        buf.add(s, a, r, s_next, done)
        s = s_next

print(f"buffer 大小: {len(buf)} / capacity {buf.capacity}")

# 采样一个 batch
batch = buf.sample(32)
states, actions, rewards, next_states, dones = batch
print(f"batch shapes:")
print(f"  states:      {states.shape}, dtype={states.dtype}")
print(f"  actions:     {actions.shape}, dtype={actions.dtype}")
print(f"  rewards:     {rewards.shape}")
print(f"  next_states: {next_states.shape}")
print(f"  dones:       {dones.shape}")

# 验证采样是均匀的：统计 buffer 中前 100 条被采到的次数
counts = np.zeros(100)
rng = np.random.default_rng(0)
for _ in range(10000):
    s_b, _, _, _, _ = buf.sample(32, rng=rng)
    # 比较前 4 维是否匹配 buffer 前 100 条
    pass
# 直接看 rewards 的分布
print(f"\nrewards 统计: mean={rewards.mean():.3f}, std={rewards.std():.3f}, min={rewards.min()}, max={rewards.max()}")
print(f"dones 比例: {dones.mean():.2%}")

## 6.6 Target Network：稳定"追逐移动目标"

### 6.6.1 问题：移动目标

如果用同一个 $Q_\theta$ 既算 prediction 又算 target：

$$
\text{target} = r + \gamma \max_{a'} Q(s', a'; \theta)
$$

每次 $\theta$ 更新，target 也跟着变。**这就像追自己尾巴**——prediction 永远追不上一个不断变化的 target。

### 6.6.2 解决方案：两份参数

维护**两个**网络：
- **Online network** $Q_\theta$：每步更新，用于行为（ε-greedy）和 prediction
- **Target network** $Q_{\theta^-}$：用于计算 target，**不直接训练**

### 6.6.3 更新 target network

两种主流方式：

**(1) Hard update（Mnih 2015 DQN 用）**：每 $N$ 步把 online 直接拷贝到 target

```python
if step % target_update_freq == 0:
    target_net.load_state_dict(online_net.state_dict())
```

**(2) Polyak（soft）update（SAC, TD3 用）**：每步让 target 慢慢追

```python
# θ⁻ ← τ · θ + (1-τ) · θ⁻
for tp, op in zip(target_net.parameters(), online_net.parameters()):
    tp.mul_(1-tau).add_(op, alpha=tau)
```

通常 $\tau = 0.001 \sim 0.01$。**$\tau \cdot N = 1$ 时和 hard update 等效**（每 $1/\tau$ 步"累积更新一次"的量）。

### 6.6.4 半梯度为什么 work（重申）

回到 6.3.3：半梯度 = target 不参与求导。**target network 让这一点在代码里实现得很自然**——把 target 计算放进 `torch.no_grad()` 块，target 用的是 $\theta^-$（独立于 $\theta$）。这就是 Ch03 §3.3 预告的"target network 是深度 RL 调参的主要动机"。

In [ ]:
# Hard update vs Polyak update 对比
torch.manual_seed(0)
online = QNetwork(4, 2, [32, 32])
target_hard = QNetwork(4, 2, [32, 32])
target_polyak = QNetwork(4, 2, [32, 32])
target_hard.load_state_dict(online.state_dict())
target_polyak.load_state_dict(online.state_dict())

# 模拟 online 经过若干步训练（手动扰动权重，避免优化发散）
with torch.no_grad():
    for p in online.parameters():
        p.add_(torch.randn_like(p) * 0.1)

# 检查扰动后 online != target
def param_diff(net1, net2):
    return float(sum((p - q).abs().sum().item() for p, q in zip(net1.parameters(), net2.parameters())))

print(f"扰动后 online vs target_hard 差异: {param_diff(online, target_hard):.4f}（应非零）")

# Hard update（一次性拷贝）
hard_update(target_hard, online)
print(f"hard_update 后:                    {param_diff(online, target_hard):.6f}（应≈0）")

# Polyak update（tau=0.01，做 100 次累积 ~ hard update）
for _ in range(100):
    polyak_update(target_polyak, online, tau=0.01)
print(f"polyak 100 步 (τ=0.01) 后:         {param_diff(online, target_polyak):.6f}（应接近 0）" )

## 6.7 DQN 完整算法 + CartPoleLite 训练

### 6.7.1 完整算法（Mnih et al. 2015）

```
初始化：
    online Q_θ（随机初始化）
    target Q_θ⁻ ← Q_θ（拷贝）
    replay buffer D（容量 N）
    ε_start = 1.0, ε_end = 0.05, ε_decay_steps = 1000

for step = 1, 2, ..., total_steps:
    1. ε-greedy 选 action（ε 随 step 线性衰减）
    2. 执行 action，观察 (s, a, r, s', done)，存入 D
    3. 从 D 随机采样 batch (s, a, r, s', done)
    4. target = r + γ · max_a' Q(s', a'; θ⁻) · (1 - done)
    5. loss = (target - Q(s, a; θ))² （半梯度 + MSE）
    6. θ ← θ - α · ∇_θ loss
    7. 每 target_update_freq 步：θ⁻ ← θ（hard update）
    8. if done: reset env
```

### 6.7.2 完整 PyTorch 实现

我们的 `utils/dqn_utils.py` 已封装好单步更新 `dqn_update_step`。下面组装完整训练循环。

In [ ]:
# 完整 DQN 训练循环
def train_dqn(
    env, online_net, target_net,
    n_total_steps=8000,
    replay_capacity=10000,
    batch_size=64,
    gamma=0.99,
    lr=1e-3,
    eps_start=1.0, eps_end=0.05, eps_decay_steps=1000,
    target_update_freq=200,
    warmup_steps=500,
    train_every=4,  # 每 4 步环境交互训练 1 次
    seed=42,
    verbose=True,
):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)

    buffer = ReplayBuffer(capacity=replay_capacity, state_dim=env.observation_dim)
    optimizer = torch.optim.Adam(online_net.parameters(), lr=lr)

    # 训练日志
    episode_rewards = []
    episode_lengths = []
    losses = []
    eps_history = []
    q_means = []

    s = env.reset()
    ep_reward = 0
    ep_length = 0
    episode = 0

    for step in range(1, n_total_steps + 1):
        # 1. ε-greedy action
        epsilon = linear_epsilon_schedule(step, eps_start, eps_end, eps_decay_steps)
        a = epsilon_greedy_action(online_net, s, epsilon, env.nA, rng=rng)

        # 2. 执行
        s_next, r, done, _ = env.step(a)
        buffer.add(s, a, r, s_next, done)
        ep_reward += r
        ep_length += 1

        s = s_next

        # 3. 训练（warmup 后开始）
        if step > warmup_steps and step % train_every == 0 and len(buffer) >= batch_size:
            batch = buffer.sample(batch_size, rng=rng)
            metrics = dqn_update_step(online_net, target_net, optimizer, batch, gamma=gamma)
            losses.append(metrics['loss'])
            q_means.append(metrics['q_mean'])

        # 4. target network update
        if step % target_update_freq == 0:
            hard_update(target_net, online_net)

        eps_history.append(epsilon)

        # 5. episode 结束
        if done:
            episode_rewards.append(ep_reward)
            episode_lengths.append(ep_length)
            if verbose and episode % 10 == 0:
                avg_r = np.mean(episode_rewards[-10:]) if episode_rewards else 0
                print(f"ep {episode:>3} step {step:>5} | avg_reward(10) {avg_r:>5.1f} | ε={epsilon:.3f}")
            s = env.reset()
            ep_reward = 0
            ep_length = 0
            episode += 1

    return {
        'episode_rewards': np.array(episode_rewards),
        'episode_lengths': np.array(episode_lengths),
        'losses': np.array(losses),
        'q_means': np.array(q_means),
        'eps_history': np.array(eps_history),
    }


# 跑训练
env = CartPoleLite(seed=0, max_steps=500)
torch.manual_seed(0)
online = QNetwork(state_dim=4, n_actions=2, hidden_dims=[128, 128])
target = QNetwork(state_dim=4, n_actions=2, hidden_dims=[128, 128])
hard_update(target, online)

print(f"网络参数量: {count_parameters(online)}")
print("开始训练...")
metrics = train_dqn(env, online, target, n_total_steps=6000, verbose=True)
print(f"\n训练结束。最后 10 episodes 平均 reward: {metrics['episode_rewards'][-10:].mean():.1f}")

In [ ]:
# 画训练曲线
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# 1. episode reward
ax = axes[0, 0]
ax.plot(metrics['episode_rewards'], alpha=0.4, label='raw')
if len(metrics['episode_rewards']) > 10:
    smoothed = np.convolve(metrics['episode_rewards'], np.ones(10)/10, mode='valid')
    ax.plot(smoothed, 'r-', linewidth=2, label='smoothed (w=10)')
ax.axhline(500, color='g', linestyle='--', alpha=0.5, label='max possible (500)')
ax.set_xlabel('episode'); ax.set_ylabel('reward')
ax.set_title('Episode reward'); ax.legend(); ax.grid(alpha=0.3)

# 2. loss
ax = axes[0, 1]
ax.plot(metrics['losses'], alpha=0.5)
ax.set_xlabel('training step'); ax.set_ylabel('MSE loss')
ax.set_title('DQN loss'); ax.set_yscale('log'); ax.grid(alpha=0.3)

# 3. Q 值均值
ax = axes[1, 0]
ax.plot(metrics['q_means'], alpha=0.5)
ax.axhline(99, color='g', linestyle='--', alpha=0.5, label='理论 Q* ≈ 99 (γ=0.99, max 500 步)')
ax.set_xlabel('training step'); ax.set_ylabel('mean Q')
ax.set_title('Q 值演化（看是否高估 → maximization bias）'); ax.legend(); ax.grid(alpha=0.3)

# 4. ε 衰减
ax = axes[1, 1]
ax.plot(metrics['eps_history'])
ax.set_xlabel('env step'); ax.set_ylabel('ε')
ax.set_title('ε schedule（线性衰减）'); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# 看训练后的策略表现（不探索，ε=0）
def evaluate(env, q_net, n_episodes=5, seed=0):
    rng = np.random.default_rng(seed)
    rewards = []
    for ep in range(n_episodes):
        s = env.reset()
        total_r = 0
        done = False
        frames = [s.copy()]
        while not done:
            with torch.no_grad():
                q = q_net(torch.as_tensor(s, dtype=torch.float32).unsqueeze(0))
            a = int(q.argmax())
            s, r, done, _ = env.step(a)
            total_r += r
            frames.append(s.copy())
        rewards.append(total_r)
        print(f"评估 ep {ep}: reward = {total_r:.0f}, 长度 = {len(frames)-1}")
    return rewards

# 跑 5 个评估 episode
print("评估（ε=0）：")
eval_rewards = evaluate(CartPoleLite(seed=0), online, n_episodes=5)
print(f"\n平均评估 reward: {np.mean(eval_rewards):.1f}（500 = 满分）")

## 6.8 Double DQN & Dueling DQN（兑现 Ch05 承诺）

### 6.8.1 回顾 Ch05 §5.8：Double Q-learning

Ch05 我们讨论过 **maximization bias**：Q-learning target 里的 $\max_a Q$ 会系统性高估（Jensen 不等式）。

**Double Q-learning 解法**：用两个独立 Q，一个选 $a^*$、一个评估。这个思路可以直接升级到 Deep RL——

### 6.8.2 Double DQN（van Hasselt et al. 2016）

DQN 用神经网络时，"两个独立 Q 网络"开销大。Double DQN 的优雅变体：

- $Q_\theta$（online）选 $a^*$
- $Q_{\theta^-}$（target）评估

```
普通 DQN:   target = r + γ · max_a Q(s', a; θ⁻)              # target 自己选 max
Double DQN: target = r + γ · Q(s', argmax_a Q(s', a; θ); θ⁻)  # online 选, target 评
```

**关键**：$\theta$ 和 $\theta^-$ 不完全相关（target 滞后）——近似独立，类似 Ch05 §5.8 的 Double Q-learning 解耦。

### 6.8.3 实现差异

`utils/dqn_utils.py` 的 `dqn_update_step` 已经支持 Double DQN——只要传 `use_double_dqn=True`。

### 6.8.4 Dueling DQN（Wang et al. 2016）

另一种架构改进：把 $Q$ 拆成 $V + A$。

$$
Q(s, a; \theta) = V(s; \theta) + A(s, a; \theta) - \frac{1}{|\mathcal{A}|} \sum_{a'} A(s, a'; \theta)
$$

**直觉**：在某些状态下，动作选择不重要（如 CartPole 杆子很稳时左/右差别不大）。Dueling 让网络更容易学到"$V$ 大但 $A$ 都接近 0"的解。

**减均值的目的**：可识别性（identifiability）——否则 $V$ 和 $A$ 可以任意平移而不改变 $Q$（$V + c, A - c$ 给出同样 $Q$）。减均值让 $\sum_a A(s, a) = 0$，唯一确定 $V$ 和 $A$ 的分解。

我们的 `DuelingQNetwork` 在 `utils/networks.py`。

In [ ]:
# 对比：标准 DQN vs Double DQN vs Dueling DQN
torch.manual_seed(0)
np.random.seed(0)

configs = {
    'DQN (standard)': dict(net_cls=QNetwork, double=False),
    'Double DQN':     dict(net_cls=QNetwork, double=True),
    'Dueling DQN':    dict(net_cls=DuelingQNetwork, double=False),
}

results = {}
for name, cfg in configs.items():
    print(f"\n--- 训练 {name} ---")
    env = CartPoleLite(seed=0, max_steps=500)
    torch.manual_seed(42)
    online = cfg['net_cls'](state_dim=4, n_actions=2, hidden_dims=[128, 128])
    target = cfg['net_cls'](state_dim=4, n_actions=2, hidden_dims=[128, 128])
    hard_update(target, online)

    # 用 double_dqn 标志重写训练循环（简化版——直接用我们的 utils）
    buffer = ReplayBuffer(capacity=10000, state_dim=4)
    opt = torch.optim.Adam(online.parameters(), lr=1e-3)
    rng = np.random.default_rng(42)

    ep_rewards = []
    s = env.reset()
    ep_r = 0
    for step in range(1, 5001):
        eps = linear_epsilon_schedule(step, 1.0, 0.05, 500)
        a = epsilon_greedy_action(online, s, eps, env.nA, rng=rng)
        s_next, r, done, _ = env.step(a)
        buffer.add(s, a, r, s_next, done)
        ep_r += r
        s = s_next

        if step > 200 and step % 4 == 0 and len(buffer) >= 64:
            batch = buffer.sample(64, rng=rng)
            dqn_update_step(online, target, opt, batch, gamma=0.99, use_double_dqn=cfg['double'])

        if step % 200 == 0:
            hard_update(target, online)

        if done:
            ep_rewards.append(ep_r)
            s = env.reset()
            ep_r = 0

    results[name] = ep_rewards
    print(f"  最后 20 ep 平均 reward: {np.mean(ep_rewards[-20:]):.1f}")

# 对比曲线
fig, ax = plt.subplots(figsize=(9, 5))
for name, rs in results.items():
    if len(rs) > 10:
        sm = np.convolve(rs, np.ones(10)/10, mode='valid')
        ax.plot(sm, label=name, linewidth=2)
ax.set_xlabel('episode'); ax.set_ylabel('reward (smoothed w=10)')
ax.set_title('三种 DQN 变种对比（CartPoleLite）')
ax.legend(); ax.grid(alpha=0.3)
ax.axhline(500, color='g', linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

## 6.9 调参经验

DQN 调参的常见旋钮和推荐起点：

| 超参 | 推荐起点 | 调参直觉 |
|---|---|---|
| **Learning rate** $\alpha$ | `1e-3`（Adam）| 太大→震荡；太小→慢。Adam 比 SGD 稳 |
| **Batch size** | `64` | 太小→噪声；太大→显存压力、单 epoch 慢 |
| **Replay capacity** | `1e4 ~ 1e5` | 太小→遗忘老经验；太大→内存 |
| **Target update freq**（hard）| `200 ~ 1000` | 太小→target 不稳；太大→学不到新东西 |
| **τ**（Polyak）| `0.001 ~ 0.01` | 与 target update freq 互补，二选一 |
| **ε schedule** | `1.0 → 0.05 线性, decay=1000 步` | 前期充分探索，后期收敛 |
| **γ** | `0.99` | CartPole 类用 0.99；长 horizon 用 0.999 |
| **Hidden layers** | `[128, 128]` | 简单任务别用太深；过深易过拟合 |
| **Gradient clipping** | `max_norm=10` | 防梯度爆炸 |

### 6.9.1 DQN 训练"不学"的常见排查

| 症状 | 可能原因 |
|---|---|
| reward 一直 ~10 | ε 太大没衰减 / 学习率太小 / 网络太小 |
| loss 一直涨 | 学习率太大 / target network 没更新 / 梯度爆炸 |
| Q 值远大于真实 | maximization bias → 试试 Double DQN |
| 学到一半崩了 | replay buffer 太小 / target update freq 太小 |

## 6.10 小结 + Ch07 预告

### 6.10.1 本章核心收获

1. **函数逼近**：从表格 $Q(s, a)$ 到参数化 $Q(s, a; \theta)$——RL 现实问题的起点
2. **梯度 TD + 半梯度**：把 TD error 转成神经网络 loss，target 放在 `no_grad`
3. **致命三件套**：bootstrap + off-policy + FA = 不稳定根源
4. **三大稳定化技巧**：
   - **Experience Replay**：打破时序相关、提高样本效率
   - **Target Network**：稳定"追逐移动目标"
   - **Double DQN**：消除 maximization bias
5. **完整 DQN 训练循环**：8 个步骤的算法，~50 行 PyTorch 代码

### 6.10.2 DQN 的局限（铺垫 Ch07）

DQN 有两个本质限制：

1. **只能处理离散动作**：架构 A 输出固定 $|\mathcal{A}|$ 维 Q，连续动作无法枚举
2. **学的是确定性策略**（argmax）：但有些任务需要随机策略（如石头剪刀布，确定性会被对手利用）

**Ch07 策略梯度定理**将直接参数化策略 $\pi_\theta(a|s)$——一举解决这两个问题。但代价是：
- **On-policy**（DQN 是 off-policy）→ 样本效率低
- **高方差**（policy gradient 的核心痛点）

### 6.10.3 与 Phase 1 兑现的承诺

| Phase 1 预告 | Ch06 是否兑现 |
|---|---|
| Experience Replay 是 DQN 核心 | ✓ §6.5 |
| Target Network 解决"追移动目标" | ✓ §6.6 |
| Double DQN = target net + Q_θ 选 + Q_θ⁻ 评 | ✓ §6.8 |
| 致命三件套引发不稳定 | ✓ §6.4 |
| DQN 用神经网络 + mini-batch | ✓ §6.7 |
| Robbins-Monro → DQN learning rate | ✓ §6.3 半梯度下降 |

---

下一章：**第 7 章 — 策略梯度定理**。
我们将放弃 Q 值，直接参数化策略 $\pi_\theta(a|s)$，并推导 RL 中最优雅的等式之一：$\nabla_\theta J(\theta) = \mathbb{E}[\nabla_\theta \log \pi(a|s) \cdot Q^\pi(s, a)]$。

## 6.11 📝 练习

### 练习 1（必做）：优先经验回放（PER 简化版）

均匀采样是 `ReplayBuffer` 的默认行为；PER 的想法是 **TD error 大的样本更值得学**。

**任务**：
1. 实现简化版 `PrioritizedReplayBuffer`：add 时记录优先级，采样按 p 的比例（`rng.choice(..., p=...)`）而不是均匀抽
2. 优先级用 |TD error| + 1e-4；刚加入、还没算过 TD error 的样本给当前最大优先级
3. 在 CartPoleLite 上对比 PER vs 均匀采样（同一 seed、同一网络），画 30-episode 平均回报曲线

<details><summary>提示</summary>

- 逐样本 TD error = |q_sa − target|，在训练循环里 `torch.no_grad()` 下算（`dqn_update_step` 返回的是均值，不够用）
- 采样概率要归一化 p_i / sum(p)；新样本给 max(p)，否则可能永远采不到
- 进阶（选做）：重要性权重 w_i = (N·p_i)^(−β)，loss 乘 w 纠正非均匀采样带来的分布偏移
</details>

**预期结果**：PER 前期收敛更快（高误差样本优先学），最终性能至少不差于均匀采样。

### 练习 2（选做）：ε 衰减方式对比

把 `linear_epsilon_schedule` 换成指数衰减 eps_end + (eps_start − eps_end)·exp(−step/tau)，tau 取 500 / 2000 各跑一次，对比学习曲线的起步速度与最终性能。想想：哪种衰减让"学到一半还在乱走"，哪种"太早贪心"？

*（开放练习，无参考答案——写完可以在 30 个 seed 下对比两条曲线的方差。）
> 📖 做完练习后，去根目录 STUDY_GUIDE.md 做 Ch06 的自测题再进入下一章。